# Advanced Assignment
In this part of the advanced-level exercises, we will explore how to use our saved model to make predictions on unseen data.

Please return a Jupyter notebook as a submission in Canvas, to make the grading easier for us.

In [2]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from tqdm import tqdm
import numpy as np
import re
from matplotlib import pyplot as plt
from nltk.corpus import treebank
from gensim.models import KeyedVectors
from keras.optimizers import Adam
# from keras.preprocessing.text import Tokenizer ## this is DEPRECATED
from keras.layers import TextVectorization
from keras.preprocessing.sequence import pad_sequences
# from keras.utils.np_utils import to_categorical
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Embedding
from keras.layers import Dense, Input
from keras.layers import TimeDistributed
from keras.layers import LSTM, GRU, SimpleRNN
from keras.models import Model
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
import pandas as pd
import nltk
import tensorflow as tf
import string
from tensorflow.keras.preprocessing.text import Tokenizer, tokenizer_from_json

### Exercise 1
Refer to Keras again on how to load your saved model, [here](https://www.tensorflow.org/guide/keras/save_and_serialize).

In [3]:
#add code here
best_model = tf.keras.models.load_model('model_gru.h5')
best_model.summary()


Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_8 (Embedding)         │ (None, 60, 32)         │       397,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ (None, 60, 60)         │        16,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, 60, 13)         │           793 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 414,835 (1.58 MB)

 Trainable params: 414,833 (1.58 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [4]:
Corpus=['The articles in this special section focus on using natural language generation techniques (NLG) and natural language processing (NLP) to build computational systems that generate reports and other kinds of text in human languages.',
        'NLG uses analytics, AI, and NLP to obtain relevant information about non-linguistic data and to generate textual summaries and explanations of these data which help people understand and benefit from them.',
        'In this regard, NLG is a research field that addresses the data-value chain by using natural language as a tool for bridging the gap between raw data and valuable information communicated to users in a comprehensible way, adapted to their information needs.']

a. We want to be able to tag the three sentences in the corpus above [source](https://ieeexplore.ieee.org/document/7983468).

Figure out the necessary steps that should precede passing the input into your **h5** model.


In [5]:
#Your code goes here
with open("x_tokenizer.json") as f:
    x_tokenizer = tokenizer_from_json(f.read())
with open("y_tokenizer.json") as f:
    y_tokenizer = tokenizer_from_json(f.read())
max_len = 60

tokenized = [nltk.word_tokenize(s) for s in Corpus]

X_encoded = x_tokenizer.texts_to_sequences(tokenized)

X_padded = pad_sequences(X_encoded, maxlen=max_len, padding="post", truncating="post")



b. Use the model to make the predictions on this dataset

In [6]:
#Your code goes here
predictions = best_model.predict(X_padded)
print(predictions.shape)
pred_ids = predictions.argmax(axis=-1)
print(pred_ids)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 383ms/step
(3, 60, 13)
[[ 5  1  4  5  7  1  1  4  2  7  1  1  1  1  1  1 12  7  1  1  1  1  1  9
   2  7  1  4  1  1 12  7  1  4  1  4  7  1  3  3  3  3  3  3  3  3  3  3
   3  3  3  3  3  3  3  3  3  3  3  3]
 [ 7  2  1  3  7  3 12  1  9  2  7  1  4  7  1 12  9  2  1  1 12  1  4  5
   1  5  2  1  1 12  2  4 11  3  3  3  3  3  3  3  3  3  3  3  3  3  3  3
   3  3  3  3  3  3  3  3  3  3  3  3]
 [ 4  5  1  3  7  2  5  1  1  4  1  5  1  1  4  2  7  1  4  5  1  4  7  5
   1  4  7  1 12  7  1  1  9  1  4  5  7  1  3  2  9 11  1  1  3  3  3  3
   3  3  3  3  3  3  3  3  3  3  3  3]]


c. Tag the corpus above

In [7]:
#Your code goes here
tagged = []
for words, p in zip(tokenized, pred_ids):
    sentence_tags = []
    for word, i in zip(words, p[:len(words)]):
        tag = y_tokenizer.index_word.get(i, 'PAD')
        sentence_tags.append((word, tag))
    tagged.append(sentence_tags);

for t in tagged:
    print(t)

[('The', 'DET'), ('articles', 'NOUN'), ('in', 'ADP'), ('this', 'DET'), ('special', 'ADJ'), ('section', 'NOUN'), ('focus', 'NOUN'), ('on', 'ADP'), ('using', 'VERB'), ('natural', 'ADJ'), ('language', 'NOUN'), ('generation', 'NOUN'), ('techniques', 'NOUN'), ('(', 'NOUN'), ('NLG', 'NOUN'), (')', 'NOUN'), ('and', 'CONJ'), ('natural', 'ADJ'), ('language', 'NOUN'), ('processing', 'NOUN'), ('(', 'NOUN'), ('NLP', 'NOUN'), (')', 'NOUN'), ('to', 'PRT'), ('build', 'VERB'), ('computational', 'ADJ'), ('systems', 'NOUN'), ('that', 'ADP'), ('generate', 'NOUN'), ('reports', 'NOUN'), ('and', 'CONJ'), ('other', 'ADJ'), ('kinds', 'NOUN'), ('of', 'ADP'), ('text', 'NOUN'), ('in', 'ADP'), ('human', 'ADJ'), ('languages', 'NOUN'), ('.', '.')]
[('NLG', 'ADJ'), ('uses', 'VERB'), ('analytics', 'NOUN'), (',', '.'), ('AI', 'ADJ'), (',', '.'), ('and', 'CONJ'), ('NLP', 'NOUN'), ('to', 'PRT'), ('obtain', 'VERB'), ('relevant', 'ADJ'), ('information', 'NOUN'), ('about', 'ADP'), ('non-linguistic', 'ADJ'), ('data', 'NOUN'

d. Save the corpus and the predictions on a file, and upload it with your submission on Canvas.

In [8]:
rows = [(sentence_idx, word, tag) for sentence_idx, sentence in enumerate(tagged) for word, tag in sentence]
df = pd.DataFrame(rows, columns=['sentence no.', 'word', 'tag'])
df.to_csv("results.csv", index=False)
df


,sentence no.,word,tag
0,0,The,DET
1,0,articles,NOUN
2,0,in,ADP
3,0,this,DET
4,0,special,ADJ
...,...,...,...
113,2,to,PRT
114,2,their,PRON
115,2,information,NOUN
116,2,needs,NOUN
